# Memory Native — LUT-GEMM scaling to multiple tokens (A100)

One-click reproduction of `research/results/lut_scaling_2026_10_17/`:

1. verify GPU / versions and clone `research/lut-multi-token` at a pinned commit;
2. run the correctness tests for the new multi-token LUT kernels;
3. run three measurement harnesses (eager sweep, CUDA-graph replay, counter-vs-AdamW);
4. plot latency vs tokens and print the headline tables.

Measured reference run: A100-SXM4-40GB, torch 2.11.0+cu130, CUDA 13.0, Triton 3.6.0,
commit `ab39b3f3b13fc663f58730e25eaf214f6ec17570`.
Read `research/results/lut_scaling_2026_10_17/REPORT.md` for the analysis.

Needs a GPU runtime (Runtime → Change runtime type → GPU).

In [ ]:
import os, subprocess, sys
print(subprocess.run(["bash","-lc","nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"],
                     capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("device:", p.name, f"{p.total_memory/2**30:.1f} GiB", p.multi_processor_count, "SMs")
try:
    import triton; print("triton", triton.__version__)
except Exception as e:
    print("triton missing:", e)

In [ ]:
# Clone the research branch and install the package (no pip deps beyond torch+triton).
COMMIT = "ab39b3f3b13fc663f58730e25eaf214f6ec17570"
subprocess.run(["bash","-lc",f"""
set -e
cd /content
[ -d memory-native ] || git clone -q https://github.com/kharkilirov1/memory-native.git
cd memory-native
git fetch -q origin research/lut-multi-token || true
git checkout -q {COMMIT}
git log --oneline -1
python -m pip install -q -e . 2>&1 | tail -1
"""], check=True)
print("cwd:", os.getcwd())

In [ ]:
# Correctness: CPU algebra + CUDA kernels (expect 16 passed on a GPU runtime).
print(subprocess.run(["bash","-lc","cd /content/memory-native && python -m pytest tests/test_lut_scaling.py -q -p no:cacheprovider 2>&1 | tail -3"],
                     capture_output=True, text=True).stdout)

In [ ]:
# 1) Eager sweep: what a loop pays per call (includes launch overhead).
OUT = "/content/results/lut_scaling_2026_10_17"
print(subprocess.run(["bash","-lc",f"""cd /content/memory-native && \
python scripts/bench_lut_scaling.py --shapes 1024,2048,4096 \
  --batches 1,2,4,8,16,32,64,128,256,512 --iters 20 --reps 3 --out-dir {OUT} 2>&1 | tail -32"""],
                     capture_output=True, text=True).stdout[-6000:])

In [ ]:
# 2) CUDA-graph replay sweep: the GPU-side ceiling (one launch per replay).
print(subprocess.run(["bash","-lc",f"""cd /content/memory-native && \
python scripts/bench_lut_graph.py --shapes 1024,4096 \
  --batches 1,2,8,32,128,512 --out-dir {OUT} 2>&1 | tail -32"""],
                     capture_output=True, text=True).stdout[-6000:])

In [ ]:
# 3) Counter updater vs AdamW: verification checks first, then phase timings.
print(subprocess.run(["bash","-lc",f"""cd /content/memory-native && \
python scripts/bench_counter_vs_adam.py --shapes 1024,2048,4096 --batch 512 \
  --iters 20 --reps 3 --out-dir {OUT} 2>&1 | grep -E 'VERIFICATION|equipped|exactly|grad_present|PHASES|shape|dense|counter|artifacts' | tail -24"""],
                     capture_output=True, text=True).stdout[-6000:])

In [ ]:
# Latency vs tokens (4096^2) from the graph CSV, plus the ratio panel.
import csv, matplotlib.pyplot as plt
OUT = "/content/results/lut_scaling_2026_10_17"
rows = list(csv.DictReader(open(f"{OUT}/graph_results.csv")))
paths = ["dense_bf16", "dense_fp32", "fused_decode", "lut_full", "lut_gather"]
ms = sorted({int(r["M"]) for r in rows})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
for shape in (1024, 4096):
    for path in paths:
        sel = {int(r["M"]): float(r["graph_ms"]) for r in rows
               if int(r["shape"]) == shape and r["path"] == path}
        xs = [m for m in ms if m in sel]
        if xs:
            ax1.loglog(xs, [sel[m] for m in xs], marker="o", label=f"{path} ({shape}^2)")
ax1.set_xlabel("tokens M"); ax1.set_ylabel("ms (CUDA graph replay)")
ax1.set_title("latency vs batch size"); ax1.grid(True, which="both", alpha=0.3)
base = {}
for r in rows:
    if int(r["shape"]) == 4096 and r["path"] == "dense_bf16":
        base[int(r["M"])] = float(r["graph_ms"])
for path in ("dense_fp32", "fused_decode", "lut_gather"):
    sel = {int(r["M"]): float(r["graph_ms"]) for r in rows
           if int(r["shape"]) == 4096 and r["path"] == path}
    xs = [m for m in ms if m in sel and m in base]
    ax2.loglog(xs, [sel[m] / base[m] for m in xs], marker="s", label=path)
ax2.axhline(1.0, color="k", ls="--", lw=1)
ax2.set_xlabel("tokens M"); ax2.set_ylabel("time / dense bf16")
ax2.set_title("4096^2: ratio to bf16 tensor-core GEMM")
ax2.grid(True, which="both", alpha=0.3)
ax1.legend(fontsize=8); ax2.legend(fontsize=8); plt.tight_layout(); plt.show()


In [ ]:
# Crossover table (4096², graph replay) and artifact locations.
import collections
by = collections.defaultdict(dict)
for r in rows:
    if int(r["shape"]) == 4096:
        by[int(r["M"])][r["path"]] = float(r["graph_ms"])
print(f"{'M':>5} {'dense_fp32':>11} {'dense_bf16':>11} {'fused':>9} {'lut_gather':>11} {'lut_vs_fp32':>11}")
for m in sorted(by):
    d = by[m]
    print(f"{m:>5} {d['dense_fp32']:>11.4f} {d['dense_bf16']:>11.4f} "
          f"{d['fused_decode']:>9.4f} {d['lut_gather']:>11.4f} {d['lut_gather']/d['dense_fp32']:>11.2f}")
print("\nartifacts under:", OUT)
print(subprocess.run(["bash","-lc",f"ls -la {OUT}"], capture_output=True, text=True).stdout)
# Optional: copy to Google Drive (mount first if you want persistence).
# from google.colab import drive; drive.mount('/content/drive')
# subprocess.run(["bash","-lc",f"mkdir -p /content/drive/MyDrive/mn_lut && cp -r {OUT}/* /content/drive/MyDrive/mn_lut/"])


## How to read the output

* `lut_gather` = the amortized serving path (weight-side index table built once, LUT built
  per token, gather). `lut_full` adds the per-call index build; the difference is the
  value of amortizing weight preprocessing.
* Eager timings are per-call costs including this VM's launch overhead; the graph timings
  are the GPU-side ceiling. Ratios are the stable quantity, absolute ms are not.
* The expected crossover on A100: LUT beats cuBLAS fp32 at M≈1–8 (4096²), loses from
  M≈32; it never beats bf16 tensor cores; it is 3–35× faster than the repo's
  fused-decode kernel at every M because that kernel pads M to 32 rows.
* Counter vs AdamW: the harness verifies real gradients/AdamW state/parameter changes
  and exactly one counter update per step before timing anything; decimated steps are a
  different algorithm, not a cheaper AdamW step.

Full analysis, VRAM accounting and limits: `research/results/lut_scaling_2026_10_17/REPORT.md`.